In [ ]:
import gdown
from sklearn.metrics import classification_report
file_id = "1vnILayXgxpPRKrgqWfQXsCrub6G4ywHR"
output = "TrafficEffNet.keras"  # Replace with a desired filename (e.g., "data.zip" or "model.h5" based on the file type)

# gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

from tensorflow.keras.models import load_model

model = load_model("model.keras")

2025-04-10 19:08:02.753890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744304882.776236  272549 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744304882.781699  272549 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744304882.797596  272549 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1744304882.797620  272549 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1744304882.797622  272549 computation_placer.cc:177] computation placer alr

In [2]:
import gdown
file_id = "1CqWWWXFpAd9vzVM8zdc4UwdAz_3d4pCn"
output = "testset_sample.zip"  # Adjust the output name based on the actual file format (e.g., .zip, .csv, .parquet, etc.)

# gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

In [3]:
import zipfile
import os

with zipfile.ZipFile("testset_sample.zip", 'r') as zip_ref:
    zip_ref.extractall("testset")  # Extract to a folder named "dataset"

# Verify the extracted contents
os.listdir("testset")

['00694.png',
 '05701.png',
 '11895.png',
 '03185.png',
 '11150.png',
 '06190.png',
 '03822.png',
 '10524.png',
 '00095.png',
 '04449.png',
 '05611.png',
 '08915.png',
 '07970.png',
 '11738.png',
 '03594.png',
 '04185.png',
 '11230.png',
 '04436.png',
 '00276.png',
 '03411.png',
 '00018.png',
 '07770.png',
 '05160.png',
 '09196.png',
 '05715.png',
 '08502.png',
 '02381.png',
 '01157.png',
 '07411.png',
 '07310.png',
 '08332.png',
 '09531.png',
 '04980.png',
 '05736.png',
 '05981.png',
 '11369.png',
 '06836.png',
 '09418.png',
 '02838.png',
 '05280.png',
 '07243.png',
 '03150.png',
 '00749.png',
 '10892.png',
 '09834.png',
 '12057.png',
 '12020.png',
 '05786.png',
 '06212.png',
 '11759.png',
 '03357.png',
 '00861.png',
 '12585.png',
 '06464.png',
 '07466.png',
 '02108.png',
 '08006.png',
 '00243.png',
 '01175.png',
 '10164.png',
 '00971.png',
 '00960.png',
 '01895.png',
 '06805.png',
 '04162.png',
 '05524.png',
 '07987.png',
 '09702.png',
 '04131.png',
 '01237.png',
 '02948.png',
 '0767

In [4]:
import pandas as pd

# Load the CSV file
labels_df = pd.read_csv("testset/Attack_Test.csv")

# Display the first few rows to understand the structure
print(labels_df.head())

# Assuming the CSV has columns like 'filename' and 'label'
# Adjust column names based on the actual CSV structure
image_filenames = labels_df['Path'].values  # e.g., '12617.png'
labels = labels_df['ClassId'].values  # e.g., 0 or 1, or class names

   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId       Path
0     39      39       6       5      34      34        0  00243.png
1     39      40       5       5      34      35        0  00778.png
2     35      36       5       6      30      31        0  04726.png
3     49      51       6       5      44      46        0  06854.png
4     34      34       6       6      29      29        0  02045.png


In [ ]:
import tensorflow as tf
import numpy as np

# Define image parameters
img_height = 240  # Adjust to match your model's expected input size
img_width = 240
batch_size = 32

# Function to load and preprocess images (using tf operations)
def load_and_preprocess_image(image_path):
    # Read the image file
    img = tf.io.read_file(image_path)  # image_path is already a tensor string
    img = tf.image.decode_png(img, channels=3)  # Assuming RGB images

    # Resize to target size
    img = tf.image.resize(img, [img_height, img_width])

    return img

# Create a dataset from the image filenames and labels
def create_dataset(image_filenames, labels):
    # Convert filenames and labels to tensors
    base_dir = "testset"
    image_paths = [os.path.join(base_dir, fname) for fname in image_filenames]
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    # Map the load_and_preprocess_image function
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=len(image_filenames))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Create the dataset
dataset = create_dataset(image_filenames, labels)

# Verify the dataset
for images, lbls in dataset.take(1):
    print("Image shape:", images.shape)
    print("Labels:", lbls)

Image shape: (32, 240, 240, 3)
Labels: tf.Tensor(
[16 39 41 42  2 36 31 32 18 33 31 11 12 36 34 16  5 29  1 33  0  4  1 37
  9 26 30  6 21 18 19  6], shape=(32,), dtype=int64)
tf.Tensor(
[[[[ 44.        42.        31.      ]
   [ 44.        42.        31.      ]
   [ 44.        42.        31.      ]
   ...
   [ 46.729164  48.32291   39.187492]
   [ 47.        49.        40.      ]
   [ 47.        49.        40.      ]]

  [[ 44.        42.        31.      ]
   [ 44.        42.        31.      ]
   [ 44.        42.        31.      ]
   ...
   [ 46.729164  48.32291   39.187492]
   [ 47.        49.        40.      ]
   [ 47.        49.        40.      ]]

  [[ 44.46875   42.625     31.625   ]
   [ 44.46875   42.625     31.625   ]
   [ 44.405273  42.561523  31.561523]
   ...
   [ 47.30371   48.606113  39.27213 ]
   [ 47.46875   49.15625   40.      ]
   [ 47.46875   49.15625   40.      ]]

  ...

  [[ 72.25003   58.78127   51.78127 ]
   [ 72.25003   58.78127   51.78127 ]
   [ 72.55048   59.

2025-04-10 16:17:01.128288: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
num_batches = len(image_filenames) // batch_size + (1 if len(image_filenames) % batch_size else 0)
print(f"Total number of batches: {num_batches}")

# Lists to store predictions and true labels
all_predicted_probs = []
all_true_labels = []

# Loop over each batch in the dataset
for i, (images, labels) in enumerate(dataset.take(num_batches)):
    print(f"Processing batch {i+1}/{num_batches}")
    
    # Make predictions for the current batch
    batch_predictions = model.predict(images, verbose=0)
    all_predicted_probs.append(batch_predictions)
    
    # Collect true labels for the current batch
    all_true_labels.extend(labels.numpy())

# Concatenate all predictions into a single array
all_predicted_probs = np.concatenate(all_predicted_probs, axis=0)
all_true_labels = np.array(all_true_labels)

Total number of batches: 17
Processing batch 1/17
Processing batch 2/17
Processing batch 3/17
Processing batch 4/17
Processing batch 5/17
Processing batch 6/17
Processing batch 7/17
Processing batch 8/17
Processing batch 9/17
Processing batch 10/17
Processing batch 11/17
Processing batch 12/17
Processing batch 13/17
Processing batch 14/17
Processing batch 15/17
Processing batch 16/17
Processing batch 17/17


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(None, 240, 240, 3))
  warnings.warn(msg)
2025-04-08 20:24:58.372113: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
all_predicted_probs = np.argmax(all_predicted_probs, axis=1)  # Convert predictions to class labels


# Calculate accuracy
print(classification_report(all_true_labels, all_predicted_probs))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        12
           3       1.00      0.92      0.96        12
           4       1.00      1.00      1.00        12
           5       0.91      0.83      0.87        12
           6       1.00      1.00      1.00        12
           7       0.75      1.00      0.86        12
           8       0.92      1.00      0.96        12
           9       1.00      1.00      1.00        12
          10       1.00      1.00      1.00        12
          11       0.92      1.00      0.96        12
          12       1.00      0.92      0.96        12
          13       1.00      1.00      1.00        12
          14       1.00      1.00      1.00        12
          15       1.00      1.00      1.00        12
          16       1.00      1.00      1.00        12
          17       1.00    

In [ ]:
from FGSM import fgsm_attack_single_image

def generate_adversarial_examples(model, dataset, epsilon=0.1):
    adversarial_examples = []
    true_labels = []
    
    for images, labels in dataset:
        # Process each image in the batch individually
        batch_adv_images = []
        for img, lbl in zip(images, labels):
            adv_img = fgsm_attack_single_image(model, img, lbl, epsilon,normalized=False)
            batch_adv_images.append(adv_img)
        
        # Convert list of tensors to a single tensor
        batch_adv_images = tf.stack(batch_adv_images)
        adversarial_examples.append(batch_adv_images)
        true_labels.extend(labels.numpy())
    
    # Concatenate all adversarial examples
    adversarial_examples = tf.concat(adversarial_examples, axis=0)
    true_labels = np.array(true_labels)
    
    return adversarial_examples, true_labels




/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(1, 240, 240, 3))
  warnings.warn(msg)
2025-04-08 16:28:25.935440: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [66]:
epsilon = 0.007
adversarial_examples_0_007, true_labels_0_007 = generate_adversarial_examples(model, dataset, epsilon)
epsilon = 0.01
adversarial_examples_0_01, true_labels_0_01 = generate_adversarial_examples(model, dataset, epsilon)

epsilon = 0.03
adversarial_examples_0_03, true_labels_0_03 = generate_adversarial_examples(model, dataset, epsilon)

epsilon = 0.1
adversarial_examples_0_1, true_labels_0_1= generate_adversarial_examples(model, dataset, epsilon)

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(1, 240, 240, 3))
  warnings.warn(msg)
2025-04-08 18:49:39.978560: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [67]:
predictions_adv_0_007 = model.predict(adversarial_examples_0_007)

predictions_adv_0_01 = model.predict(adversarial_examples_0_01)
predictions_adv_0_03 = model.predict(adversarial_examples_0_03)
predictions_adv_0_1 = model.predict(adversarial_examples_0_1)

print("Classification report for epsilon=0.007")
print(classification_report(true_labels_0_007, np.argmax(predictions_adv_0_007, axis=1)))
print("Classification report for epsilon=0.01")
print(classification_report(true_labels_0_01, np.argmax(predictions_adv_0_01, axis=1)))
print("Classification report for epsilon=0.03")
print(classification_report(true_labels_0_03, np.argmax(predictions_adv_0_03, axis=1)))
print("Classification report for epsilon=0.1")
print(classification_report(true_labels_0_1, np.argmax(predictions_adv_0_1, axis=1)))

17/17 ━━━━━━━━━━━━━━━━━━━━ 94s 6s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 96s 6s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 94s 6s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 90s 5s/step
Classification report for epsilon=0.007
              precision    recall  f1-score   support

           0       0.64      0.58      0.61        12
           1       0.53      0.83      0.65        12
           2       1.00      0.42      0.59        12
           3       0.56      0.42      0.48        12
           4       0.48      0.83      0.61        12
           5       0.33      0.58      0.42        12
           6       0.06      0.08      0.07        12
           7       0.40      0.50      0.44        12
           8       0.62      0.42      0.50        12
           9       0.80      0.33      0.47        12
          10       0.80      0.33      0.47        12
          11       0.32      0.83      0.47        12
          12       0.90      0.75      0.82        12
          13       0.75      1.00      0.86    

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [69]:
# Define output directory
output_dir = "adversarial_images"
os.makedirs(output_dir, exist_ok=True)

# Dictionary to map epsilon values to their adversarial examples and labels
epsilon_data = {
    0.007: {'examples': adversarial_examples_0_007, 'labels': true_labels_0_007},
    0.01: {'examples': adversarial_examples_0_01, 'labels': true_labels_0_01},
    0.03: {'examples': adversarial_examples_0_03, 'labels': true_labels_0_03},
    0.1: {'examples': adversarial_examples_0_1, 'labels': true_labels_0_1}
}

# Save adversarial examples as images and prepare CSV data
csv_data = []
for epsilon, data in epsilon_data.items():
    epsilon_dir = os.path.join(output_dir, f"eps_{epsilon}")
    os.makedirs(epsilon_dir, exist_ok=True)
    
    adv_examples = data['examples'].numpy()
    true_labels = data['labels']
    
    for i, adv_img in enumerate(adv_examples):
        # Convert to uint8 and save as PNG
        adv_img = (adv_img).astype(np.uint8)  # Adjust for normalized=False
        filename = os.path.join(f"eps_{epsilon}", f"adv_{i}.png")
        tf.keras.preprocessing.image.save_img(
            os.path.join(output_dir, filename), adv_img
        )
        csv_data.append({'epsilon': epsilon, 'index': i, 'filename': filename, 'label': true_labels[i]})

# Save labels and filenames to CSV
csv_df = pd.DataFrame(csv_data)
csv_df.to_csv(os.path.join(output_dir, "adversarial_labels.csv"), index=False)

print(f"Adversarial images saved to {output_dir} and labels to adversarial_labels.csv")

Adversarial images saved to adversarial_images and labels to adversarial_labels.csv


In [8]:
from PGD import pgd_attack_single_image


def generate_adversarial_examples_pgd(model, dataset, epsilon=0.1, alpha=0.001, iterations=10):
    adversarial_examples = []
    true_labels = []
    
    batch_count = 0  # for logging
    
    for images, labels in dataset:
        batch_count += 1
        batch_adv_images = []

        for img, lbl in zip(images, labels):
            adv_img = pgd_attack_single_image(model, img, lbl, epsilon, alpha, iterations, normalized=False)
            batch_adv_images.append(adv_img)
        
        # Convert list of tensors to a single tensor
        batch_adv_images = tf.stack(batch_adv_images)
        adversarial_examples.append(batch_adv_images)
        true_labels.extend(labels.numpy())
        
        print(f"[INFO] Finished batch {batch_count} — {len(images)} images processed")

    # Concatenate all adversarial examples
    adversarial_examples = tf.concat(adversarial_examples, axis=0)
    true_labels = np.array(true_labels)
    
    print(f"[INFO] All batches processed — Total examples: {adversarial_examples.shape[0]}")
    return adversarial_examples, true_labels


In [51]:
epsilon= [0.01,0.03,0.05,0.1]
alpha = [e / 4 for e in epsilon]
itertations = 3
print(alpha)
adversarial_examples_0_007_pgd ,true_labels_0_007_pgd = generate_adversarial_examples_pgd(model, dataset, epsilon[0], alpha[0],itertations)
adversarial_examples_0_01_pgd ,true_labels_0_01_pgd = generate_adversarial_examples_pgd(model, dataset, epsilon[1], alpha[1],itertations)
adversarial_examples_0_03_pgd ,true_labels_0_03_pgd = generate_adversarial_examples_pgd(model, dataset, epsilon[2], alpha[2],itertations)
adversarial_examples_0_1_pgd ,true_labels_0_1_pgd = generate_adversarial_examples_pgd(model, dataset, epsilon[3], alpha[3],itertations)

[0.0025, 0.0075, 0.0125, 0.025]


/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(1, 240, 240, 3))
  warnings.warn(msg)


[INFO] Finished batch 1 — 32 images processed
[INFO] Finished batch 2 — 32 images processed
[INFO] Finished batch 3 — 32 images processed
[INFO] Finished batch 4 — 32 images processed
[INFO] Finished batch 5 — 32 images processed
[INFO] Finished batch 6 — 32 images processed
[INFO] Finished batch 7 — 32 images processed
[INFO] Finished batch 8 — 32 images processed
[INFO] Finished batch 9 — 32 images processed
[INFO] Finished batch 10 — 32 images processed
[INFO] Finished batch 11 — 32 images processed
[INFO] Finished batch 12 — 32 images processed
[INFO] Finished batch 13 — 32 images processed
[INFO] Finished batch 14 — 32 images processed
[INFO] Finished batch 15 — 32 images processed
[INFO] Finished batch 16 — 32 images processed
[INFO] Finished batch 17 — 4 images processed
[INFO] All batches processed — Total examples: 516
[INFO] Finished batch 1 — 32 images processed
[INFO] Finished batch 2 — 32 images processed
[INFO] Finished batch 3 — 32 images processed
[INFO] Finished batch 

2025-04-10 17:34:36.929531: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[INFO] Finished batch 1 — 32 images processed
[INFO] Finished batch 2 — 32 images processed
[INFO] Finished batch 3 — 32 images processed
[INFO] Finished batch 4 — 32 images processed
[INFO] Finished batch 5 — 32 images processed
[INFO] Finished batch 6 — 32 images processed
[INFO] Finished batch 7 — 32 images processed
[INFO] Finished batch 8 — 32 images processed
[INFO] Finished batch 9 — 32 images processed
[INFO] Finished batch 10 — 32 images processed
[INFO] Finished batch 11 — 32 images processed
[INFO] Finished batch 12 — 32 images processed
[INFO] Finished batch 13 — 32 images processed
[INFO] Finished batch 14 — 32 images processed
[INFO] Finished batch 15 — 32 images processed
[INFO] Finished batch 16 — 32 images processed
[INFO] Finished batch 17 — 4 images processed
[INFO] All batches processed — Total examples: 516
[INFO] Finished batch 1 — 32 images processed
[INFO] Finished batch 2 — 32 images processed
[INFO] Finished batch 3 — 32 images processed
[INFO] Finished batch 

In [54]:
from sklearn.metrics import classification_report
predicted_adv_0_007_pgd = model.predict(adversarial_examples_0_007_pgd)
predicted_adv_0_01_pgd = model.predict(adversarial_examples_0_01_pgd)
predicted_adv_0_03_pgd = model.predict(adversarial_examples_0_03_pgd)
predicted_adv_0_1_pgd = model.predict(adversarial_examples_0_1_pgd)
print("Classification report for epsilon=0.007")
print(classification_report(true_labels_0_007_pgd, np.argmax(predicted_adv_0_007_pgd, axis=1)))
print("Classification report for epsilon=0.01")
print(classification_report(true_labels_0_01_pgd, np.argmax(predicted_adv_0_01_pgd, axis=1)))
print("Classification report for epsilon=0.03")
print(classification_report(true_labels_0_03_pgd, np.argmax(predicted_adv_0_03_pgd, axis=1)))
print("Classification report for epsilon=0.1")
print(classification_report(true_labels_0_1_pgd, np.argmax(predicted_adv_0_1_pgd, axis=1)))

17/17 ━━━━━━━━━━━━━━━━━━━━ 68s 4s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 65s 4s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 65s 4s/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 66s 4s/step
Classification report for epsilon=0.007
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.71      0.83      0.77        12
           2       0.43      0.25      0.32        12
           3       0.00      0.00      0.00        12
           4       0.16      0.42      0.23        12
           5       0.07      0.08      0.08        12
           6       0.20      0.08      0.12        12
           7       0.04      0.08      0.05        12
           8       0.20      0.25      0.22        12
           9       0.40      0.50      0.44        12
          10       0.75      0.25      0.38        12
          11       0.40      0.50      0.44        12
          12       0.60      0.25      0.35        12
          13       0.86      1.00      0.92    

/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/youssef-abuzeid/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [55]:
# Define output directory
output_dir = "adversarial_images_PGD"
os.makedirs(output_dir, exist_ok=True)

# Dictionary to map epsilon values to their adversarial examples and labels
epsilon_data = {
    0.007: {'examples': adversarial_examples_0_007_pgd, 'labels': true_labels_0_007_pgd},
    0.01: {'examples': adversarial_examples_0_01_pgd, 'labels': true_labels_0_01_pgd},
    0.03: {'examples': adversarial_examples_0_03_pgd, 'labels': true_labels_0_03_pgd},
    0.1: {'examples': adversarial_examples_0_1_pgd, 'labels': true_labels_0_1_pgd}
}

# Save adversarial examples as images and prepare CSV data
csv_data = []
for epsilon, data in epsilon_data.items():
    epsilon_dir = os.path.join(output_dir, f"eps_{epsilon}")
    os.makedirs(epsilon_dir, exist_ok=True)
    
    adv_examples = data['examples'].numpy()
    true_labels = data['labels']
    
    for i, adv_img in enumerate(adv_examples):
        # Convert to uint8 and save as PNG
        adv_img = (adv_img).astype(np.uint8)  # Adjust for normalized=False
        filename = os.path.join(f"eps_{epsilon}", f"adv_{i}.png")
        tf.keras.preprocessing.image.save_img(
            os.path.join(output_dir, filename), adv_img
        )
        csv_data.append({'epsilon': epsilon, 'index': i, 'filename': filename, 'label': true_labels[i]})

# Save labels and filenames to CSV
csv_df = pd.DataFrame(csv_data)
csv_df.to_csv(os.path.join(output_dir, "adversarial_labels.csv"), index=False)

print(f"Adversarial images saved to {output_dir} and labels to adversarial_labels.csv")

Adversarial images saved to adversarial_images_PGD and labels to adversarial_labels.csv


In [15]:
# GTSRB Class ID to Sign Name Mapping (43 classes)
GTSRB_CLASSES = {
    0: "Speed limit 20",
    1: "Speed limit 30",
    2: "Speed limit 50",
    3: "Speed limit 60",
    4: "Speed limit 70",
    5: "Speed limit 80",
    6: "End of speed limit 80",
    7: "Speed limit 100",
    8: "Speed limit 120",
    9: "No passing",
    10: "No passing for vehicles over 3.5 tons",
    11: "Right-of-way at next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve left",
    20: "Dangerous curve right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing for vehicles over 3.5 tons"
}


def predict_traffic_sign(img_input, model, class_mapping=GTSRB_CLASSES):
    """
    Accepts either:
    - Tensor (rank 3/4)
    - Bytes (raw image bytes)
    - File path (string)
    """

    prop = model.predict(img_input)
    print(prop.shape)
    class_id = np.argmax(prop)
    class_name = class_mapping[class_id]
    return  class_name, prop[0][class_id]

In [24]:
import tensorflow as tf

def pgd_attack_single_image(model, image, label, epsilon, alpha, iterations, normalized=True):
    """
    Perform PGD attack on a single image without normalization.
    
    Args:
        model (tf.keras.Model): Trained model.
        image (tf.Tensor): Input image of shape (H, W, C), values in [0, 255].
        label (tf.Tensor): True label (integer).
        epsilon (float): Maximum perturbation magnitude.
        alpha (float): Step size for each iteration.
        iterations (int): Number of PGD iterations.
        normalized (bool): For normalized images, set to True.

    Returns:
        tf.Tensor: Adversarial example in [0, 255].
    """
    # Adjust epsilon for non normalized images
    if not normalized:
        epsilon = epsilon * 255.0
        alpha =alpha*255.0
    # Ensure the image has a batch dimension
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension
    label = tf.expand_dims(label, axis=0)  # Add batch dimension

    adversarial_image = image
    for _ in range(iterations):
        with tf.GradientTape() as tape:
            tape.watch(adversarial_image)
            # Forward pass
            prediction = model(adversarial_image, training=False)
            # Calculate loss
            loss =  tf.keras.losses.sparse_categorical_crossentropy(label, prediction)
        
        # Calculate gradient of the loss w.r.t. the adversarial image
        gradient = tape.gradient(loss, adversarial_image)

        # Get the sign of the gradient and update the adversarial image
        adversarial_image = adversarial_image  + alpha * tf.sign(gradient)

        # Project the adversarial image into the epsilon-ball and clip to [0, 255]
        adversarial_image = tf.clip_by_value(adversarial_image, image - epsilon, image + epsilon)
        if normalized:
            adversarial_image = tf.clip_by_value(adversarial_image, 0, 1)
        else:
            adversarial_image = tf.clip_by_value(adversarial_image, 0, 255)

    return tf.squeeze(adversarial_image)


In [25]:
from PIL import Image
import numpy as np
# take = dataset.take(1)
# img =take.get_single_element()
# label = take.get_single_element()
# img = img[0][0]

# print(img.shape)
# label = take.get_single_element()
# label = label[0][0]
def load_ppm_image(image_path, target_size=(240, 240)):
    # Open the .ppm image using Pillow
    with Image.open(image_path) as img:
        # Convert image to RGB (if not already)
        img = img.convert("RGB")
        # Resize the image to the target size (now 240x240)
        img = img.resize(target_size)
        # Convert image to numpy array
        img_array = np.array(img)
        # Convert numpy array to a TensorFlow tensor and cast to float32
        img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)
        # img_tensor = tf.expand_dims(img_tensor, axis=0)
    return img_tensor

img = load_ppm_image("GTSRB_Directory/Train/5/00005_00000_00005.png") 
adv_img = pgd_attack_single_image(model, img, 0, 0.1, 0.025,20 , normalized=False)




/home/youssef-abuzeid/.local/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['Input_Layer']
Received: inputs=Tensor(shape=(1, 240, 240, 3))
  warnings.warn(msg)


## One Pixel Attack

## First load the dataset as 32x32 images to be more efficient

In [5]:
import tensorflow as tf
import numpy as np

# Define image parameters
img_height = 32  # Adjust to match your model's expected input size
img_width = 32
batch_size = 32

# Function to load and preprocess images (using tf operations)
def load_and_preprocess_image(image_path):
    # Read the image file
    img = tf.io.read_file(image_path)  # image_path is already a tensor string
    img = tf.image.decode_png(img, channels=3)  # Assuming RGB images

    # Resize to target size
    img = tf.image.resize(img, [img_height, img_width])

    return img

# Create a dataset from the image filenames and labels
def create_dataset(image_filenames, labels):
    # Convert filenames and labels to tensors
    base_dir = "testset"
    image_paths = [os.path.join(base_dir, fname) for fname in image_filenames]
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    # Map the load_and_preprocess_image function
    dataset = dataset.map(
        lambda path, label: (load_and_preprocess_image(path), label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=len(image_filenames))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Create the dataset
dataset = create_dataset(image_filenames, labels)

# Verify the dataset
for images, lbls in dataset.take(1):
    print("Image shape:", images.shape)
    print("Labels:", lbls)
# Example of using the dataset with the model
    print(images)

Image shape: (32, 32, 32, 3)
Labels: tf.Tensor(
[39  5 38 33 13  6 22 31 26 13 16 34 30 34 19  1 33 40 30 33 19 32 29  0
  9 15 31 27 22 37 18 30], shape=(32,), dtype=int64)
tf.Tensor(
[[[[ 49.15381   49.74756   39.959717]
   [ 44.198975  45.796875  39.546875]
   [ 43.638916  43.203125  37.704834]
   ...
   [189.5835   187.02612  183.94629 ]
   [157.35522  154.28125  150.79688 ]
   [ 81.12085   73.52466   69.484375]]

  [[ 57.4729    56.3479    44.113525]
   [ 47.068604  46.881104  39.506104]
   [ 37.239258  38.692383  34.525635]
   ...
   [199.29199  195.65674  196.37524 ]
   [169.38135  161.22192  157.59375 ]
   [ 85.59375   77.46875   74.24585 ]]

  [[ 45.708252  44.909668  39.567627]
   [ 47.145996  44.833496  37.744873]
   [ 45.477295  45.477295  39.36792 ]
   ...
   [194.4602   193.85938  190.33667 ]
   [176.96338  164.14038  160.00513 ]
   [ 88.381836  80.75342   75.72388 ]]

  ...

  [[166.64941  175.79175  183.448   ]
   [166.58838  170.9895   176.90088 ]
   [162.91187  170.19

2025-04-10 19:08:18.125331: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Preprocessing function to predict the labels

In [6]:
def preprocess_image(image):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (240, 240))

    return image

In [9]:
from onePixel import perform_one_pixel_attack, apply_attack


def generate_adversarial_examples_one_pixel(model, dataset, preprocess, maxiter=75, popsize=400, tol=1e-5, 
                                          attack_type='untargeted', target_labels=None):
    """
    Generate adversarial examples for a dataset using the one-pixel attack.
    
    Parameters:
        model (tf.keras.Model): Pretrained model for classification.
        dataset (tf.data.Dataset): Dataset containing batches of images and labels (images in [0, 255]).
        preprocess (callable): Function to preprocess images before model prediction.
        maxiter (int): Maximum iterations for differential evolution.
        popsize (int): Population size for differential evolution.
        tol (float): Convergence tolerance for optimization.
        attack_type (str): Type of attack ('untargeted' or 'targeted').
        target_labels (np.ndarray or None): Target labels for targeted attacks (shape (N,) if targeted).
    
    Returns:
        adversarial_examples (np.ndarray): Array of adversarial examples in [0, 255].
        true_labels (np.ndarray): Array of true labels.
    """
    adversarial_examples = []
    true_labels = []
    
    batch_count = 0  # for logging
    
    for images, labels in dataset:
        batch_count += 1
        batch_adv_images = []

        for img, lbl in zip(images.numpy(), labels.numpy()):
            # Ensure image is numpy array and in [0, 255]
            img = img.astype(np.float32)
            
            # Determine target label if targeted attack
            current_target = None
            if attack_type == 'targeted':
                if target_labels is None or len(target_labels) <= (batch_count - 1) * len(images) + len(batch_adv_images):
                    raise ValueError("Target labels must be provided for targeted attack and match dataset size.")
                current_target = target_labels[(batch_count - 1) * len(images) + len(batch_adv_images)]

            # Perform the one-pixel attack using your method
            result = perform_one_pixel_attack(
                img, model, preprocess, lbl,
                maxiter=maxiter, popsize=popsize, tol=tol,
                attack_type=attack_type, target_label=current_target
            )
            
            # Apply the perturbation using your method
            adv_img = apply_attack(img, result.x)
            batch_adv_images.append(adv_img)
            print(f"Processed image with label {lbl} — Adversarial example generated")
        
        # Convert list of numpy arrays to a single numpy array
        batch_adv_images = np.stack(batch_adv_images)
        adversarial_examples.append(batch_adv_images)
        true_labels.extend(labels.numpy())
        
        print(f"[INFO] Finished batch {batch_count} — {len(images)} images processed")

    # Concatenate all adversarial examples
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)
    true_labels = np.array(true_labels)
    
    print(f"[INFO] All batches processed — Total examples: {adversarial_examples.shape[0]}")
    return adversarial_examples, true_labels

In [12]:
adversarial_examples_one_pixel ,true_labels_one_pixel = generate_adversarial_examples_one_pixel(model, dataset, preprocess_image, maxiter=3, popsize=25, tol=1e-5)

Processed image with label 14 — Adversarial example generated
Processed image with label 19 — Adversarial example generated
Processed image with label 20 — Adversarial example generated
Processed image with label 34 — Adversarial example generated
Processed image with label 5 — Adversarial example generated
Processed image with label 23 — Adversarial example generated
Processed image with label 42 — Adversarial example generated
Processed image with label 20 — Adversarial example generated
Processed image with label 2 — Adversarial example generated
Processed image with label 25 — Adversarial example generated
Processed image with label 31 — Adversarial example generated
Processed image with label 14 — Adversarial example generated
Processed image with label 31 — Adversarial example generated
Processed image with label 40 — Adversarial example generated
Processed image with label 1 — Adversarial example generated
Processed image with label 31 — Adversarial example generated
Processed i

2025-04-11 06:07:49.738458: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [19]:
from sklearn.metrics import classification_report
print(adversarial_examples_one_pixel.shape)
resized_images = tf.image.resize(adversarial_examples_one_pixel, [240, 240], method='bilinear')
resized_images = resized_images.numpy()
print(resized_images.shape)
predictions_adv_one_pixel = model.predict(resized_images)
print(classification_report(true_labels_one_pixel, np.argmax(predictions_adv_one_pixel, axis=1)))

(516, 32, 32, 3)
(516, 240, 240, 3)
17/17 ━━━━━━━━━━━━━━━━━━━━ 77s 4s/step
              precision    recall  f1-score   support

           0       1.00      0.50      0.67        12
           1       0.79      0.92      0.85        12
           2       0.86      0.50      0.63        12
           3       0.78      0.58      0.67        12
           4       0.50      0.92      0.65        12
           5       0.38      0.50      0.43        12
           6       1.00      0.75      0.86        12
           7       0.45      0.83      0.59        12
           8       0.62      0.67      0.64        12
           9       1.00      0.92      0.96        12
          10       1.00      0.92      0.96        12
          11       0.67      1.00      0.80        12
          12       1.00      0.92      0.96        12
          13       1.00      1.00      1.00        12
          14       1.00      1.00      1.00        12
          15       1.00      0.67      0.80        12
      

In [20]:
import numpy as np
import os
from PIL import Image
import csv

# Assuming these are already defined from your snippet
# resized_images (shape: (516, 240, 240, 3)), true_labels_one_pixel

# Ensure resized_images is in the correct format (uint8, [0, 255])
if resized_images.dtype != np.uint8:
    resized_images = resized_images.astype(np.uint8)

# 1. Save each resized adversarial image as a PNG file
output_dir = 'adversarial_images_one_pixel'
os.makedirs(output_dir, exist_ok=True)

image_filenames = []  # To store filenames for CSV

for i, (img, label) in enumerate(zip(resized_images, true_labels_one_pixel)):
    # Convert the image to uint8 if not already
    img = img.astype(np.uint8)
    
    # Save as PNG with a simple index-based filename
    filename = f'{output_dir}/adv_image_{i}.png'
    img_pil = Image.fromarray(img)
    img_pil.save(filename)
    image_filenames.append(f'adv_image_{i}.png')  # Store filename for CSV
    print(f"Saved image {i} as {filename}")

# 2. Save labels and corresponding image filenames to a CSV file
with open('labels_and_images.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Image_Filename', 'True_Label'])  # Header
    for filename, label in zip(image_filenames, true_labels_one_pixel):
        writer.writerow([filename, label])
print("Saved labels and image filenames to 'labels_and_images.csv'")

# 3. Verify the number of saved files and labels
print(f"Total images saved: {len(image_filenames)}")
print(f"Total labels saved: {len(true_labels_one_pixel)}")
assert len(image_filenames) == len(true_labels_one_pixel), "Mismatch between number of images and labels"

Saved image 0 as adversarial_images_one_pixel/adv_image_0.png
Saved image 1 as adversarial_images_one_pixel/adv_image_1.png
Saved image 2 as adversarial_images_one_pixel/adv_image_2.png
Saved image 3 as adversarial_images_one_pixel/adv_image_3.png
Saved image 4 as adversarial_images_one_pixel/adv_image_4.png
Saved image 5 as adversarial_images_one_pixel/adv_image_5.png
Saved image 6 as adversarial_images_one_pixel/adv_image_6.png
Saved image 7 as adversarial_images_one_pixel/adv_image_7.png
Saved image 8 as adversarial_images_one_pixel/adv_image_8.png
Saved image 9 as adversarial_images_one_pixel/adv_image_9.png
Saved image 10 as adversarial_images_one_pixel/adv_image_10.png
Saved image 11 as adversarial_images_one_pixel/adv_image_11.png
Saved image 12 as adversarial_images_one_pixel/adv_image_12.png
Saved image 13 as adversarial_images_one_pixel/adv_image_13.png
Saved image 14 as adversarial_images_one_pixel/adv_image_14.png
Saved image 15 as adversarial_images_one_pixel/adv_image_15.

# Adversarial Attacks Comparison

## Overall Accuracy Comparison

| Attack Method | Parameters | Accuracy | Original Model |
|---------------|------------|----------|----------------|
| **Original Model** | - | 96% | - |
| **FGSM** | ε = 0.007 | 58% | 96% |
| **FGSM** | ε = 0.01 | 44% | 96% |
| **FGSM** | ε = 0.03 | 14% | 96% |
| **FGSM** | ε = 0.1 | 6% | 96% |
| **PGD** | ε = 0.007, α = 0.00175 | 35% | 96% |
| **PGD** | ε = 0.01, α = 0.0025 | 7% | 96% |
| **PGD** | ε = 0.03, α = 0.0075 | 3% | 96% |
| **PGD** | ε = 0.1, α = 0.025 | 0% | 96% |
| **One Pixel** | - | 79% | 96% |

## Detailed Performance Metrics

### Original Model Performance (No Attack)
- **Accuracy**: 96%
- **Macro Avg Precision**: 97%
- **Macro Avg Recall**: 96%
- **Macro Avg F1-Score**: 96%

### Fast Gradient Sign Method (FGSM)

#### ε = 0.007
- **Accuracy**: 58%
- **Macro Avg Precision**: 68%
- **Macro Avg Recall**: 58%
- **Macro Avg F1-Score**: 58%

#### ε = 0.01
- **Accuracy**: 44%
- **Macro Avg Precision**: 52%
- **Macro Avg Recall**: 44%
- **Macro Avg F1-Score**: 42%

#### ε = 0.03
- **Accuracy**: 14%
- **Macro Avg Precision**: 14%
- **Macro Avg Recall**: 14%
- **Macro Avg F1-Score**: 13%

#### ε = 0.1
- **Accuracy**: 6%
- **Macro Avg Precision**: 5%
- **Macro Avg Recall**: 6%
- **Macro Avg F1-Score**: 5%

### Projected Gradient Descent (PGD)

#### ε = 0.007, α = 0.00175
- **Accuracy**: 35%
- **Macro Avg Precision**: 36%
- **Macro Avg Recall**: 35%
- **Macro Avg F1-Score**: 33%

#### ε = 0.01, α = 0.0025
- **Accuracy**: 7%
- **Macro Avg Precision**: 7%
- **Macro Avg Recall**: 7%
- **Macro Avg F1-Score**: 6%

#### ε = 0.03, α = 0.0075
- **Accuracy**: 3%
- **Macro Avg Precision**: 2%
- **Macro Avg Recall**: 3%
- **Macro Avg F1-Score**: 2%

#### ε = 0.1, α = 0.025
- **Accuracy**: 0%
- **Macro Avg Precision**: 0%
- **Macro Avg Recall**: 0%
- **Macro Avg F1-Score**: 0%

### One Pixel Attack
- **Accuracy**: 79%
- **Macro Avg Precision**: 84%
- **Macro Avg Recall**: 79%
- **Macro Avg F1-Score**: 78%

## Key Observations

1. **Attack Parameters**: For PGD attacks, the step size (α) was set to ε/4, creating a more controlled iterative approach compared to FGSM's single-step perturbation.

2. **Attack Effectiveness Ranking**:
   - **PGD**: Most effective, with accuracy dropping to 35% even at the lowest perturbation (ε = 0.007, α = 0.00175)
   - **FGSM**: Moderately effective, with accuracy of 58% at ε = 0.007
   - **One Pixel**: Least effective, with accuracy remaining at 79%

3. **PGD vs FGSM Comparison**: PGD consistently outperforms FGSM in attack success at every epsilon value, demonstrating the advantage of iterative optimization approaches for finding adversarial examples.

4. **Attack Progression with Increasing Perturbation**: 
   - FGSM: 96% → 58% → 44% → 14% → 6%
   - PGD: 96% → 35% → 7% → 3% → 0%

5. **Complete Vulnerability**: At ε = 0.1 (α = 0.025), PGD completely breaks the model with 0% accuracy, while FGSM still allows 6% correct classifications.